# 02 - Feature Engineering, Model Training & Experiment Tracking

Builds the preprocessing pipeline, trains and tunes Logistic Regression and Random Forest with `GridSearchCV`, evaluates them (accuracy, precision, recall, F1, ROC-AUC), plots confusion matrices and ROC curves, logs everything to MLflow, and saves the best pipeline.

## Setup

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay)
from src.data import load_clean
from src.preprocessing import build_preprocessor
from src.train import candidate_models, evaluate
from src.config import FEATURES, TARGET, RANDOM_STATE

/home/ubuntu/heart-disease-mlops/.venv/lib/python3.10/site-packages/mlflow/protos/service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service


## 1. Feature engineering (preprocessing pipeline)
A `ColumnTransformer`: median-impute + scale numerics, most-frequent-impute + one-hot encode categoricals. The same object is reused at inference.

In [2]:
df = load_clean()
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print('train:', X_train.shape, 'test:', X_test.shape)
build_preprocessor()

train: (242, 13) test: (61, 13)


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='median')),
                                                 ('scale', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalach',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['sex', 'cp', 'fbs', 'restecg', 'exang',
                                  'slope', 'ca', 'thal'])])

## 2. Train + tune two models with GridSearchCV (5-fold, ROC-AUC)

In [3]:
results, fitted = {}, {}
for name, (estimator, grid) in candidate_models().items():
    pipe = Pipeline([('preprocess', build_preprocessor()), ('model', estimator)])
    search = GridSearchCV(pipe, grid, cv=5, scoring='roc_auc', n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search.best_estimator_
    m = evaluate(search.best_estimator_, X_test, y_test)
    m['cv_roc_auc_best'] = search.best_score_
    m['best_params'] = search.best_params_
    results[name] = m
    print(f'{name}: {m}')
pd.DataFrame(results).T

logistic_regression: {'accuracy': 0.8852459016393442, 'precision': 0.8387096774193549, 'recall': 0.9285714285714286, 'f1': 0.8813559322033898, 'roc_auc': 0.9653679653679654, 'cv_roc_auc_best': 0.8981779894823372, 'best_params': {'model__C': 0.1}}


random_forest: {'accuracy': 0.8360655737704918, 'precision': 0.8214285714285714, 'recall': 0.8214285714285714, 'f1': 0.8214285714285714, 'roc_auc': 0.9437229437229436, 'cv_roc_auc_best': 0.9028720876546965, 'best_params': {'model__max_depth': 4, 'model__n_estimators': 100}}


,accuracy,precision,recall,f1,roc_auc,cv_roc_auc_best,best_params
logistic_regression,0.885246,0.83871,0.928571,0.881356,0.965368,0.898178,{'model__C': 0.1}
random_forest,0.836066,0.821429,0.821429,0.821429,0.943723,0.902872,"{'model__max_depth': 4, 'model__n_estimators':..."


## 3. Detailed classification reports

In [4]:
for name, model in fitted.items():
    print('='*60)
    print(name)
    print(classification_report(y_test, model.predict(X_test),
          target_names=['No disease','Disease']))

logistic_regression
              precision    recall  f1-score   support

  No disease       0.93      0.85      0.89        33
     Disease       0.84      0.93      0.88        28

    accuracy                           0.89        61
   macro avg       0.89      0.89      0.89        61
weighted avg       0.89      0.89      0.89        61

random_forest
              precision    recall  f1-score   support

  No disease       0.85      0.85      0.85        33
     Disease       0.82      0.82      0.82        28

    accuracy                           0.84        61
   macro avg       0.83      0.83      0.83        61
weighted avg       0.84      0.84      0.84        61



## 4. Confusion matrices & ROC curves

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, fitted.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

/tmp/ipykernel_7172/3692362816.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, model in fitted.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)
ax.set_title('ROC curves'); plt.show()

/tmp/ipykernel_7172/2281370338.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title('ROC curves'); plt.show()


## 5. Experiment tracking + save best model
`src.train.train()` re-runs the tuning loop, logs params/metrics/plots/model to MLflow, and saves the best pipeline to `models/model.joblib`.

In [7]:
from src.train import train
summary = train()
summary

2026/07/10 09:00:42 INFO mlflow.tracking.fluent: Experiment with name 'heart-disease-classification' does not exist. Creating a new experiment.


[logistic_regression] best_params={'model__C': 0.1} {'accuracy': 0.8852459016393442, 'precision': 0.8387096774193549, 'recall': 0.9285714285714286, 'f1': 0.8813559322033898, 'roc_auc': 0.9653679653679654, 'cv_roc_auc_best': 0.8981779894823372}


[random_forest] best_params={'model__max_depth': 4, 'model__n_estimators': 100} {'accuracy': 0.8360655737704918, 'precision': 0.8214285714285714, 'recall': 0.8214285714285714, 'f1': 0.8214285714285714, 'roc_auc': 0.9437229437229436, 'cv_roc_auc_best': 0.9028720876546965}

Best model: random_forest -> saved to /home/ubuntu/heart-disease-mlops/models/model.joblib


{'best_model': 'random_forest',
 'accuracy': 0.8360655737704918,
 'precision': 0.8214285714285714,
 'recall': 0.8214285714285714,
 'f1': 0.8214285714285714,
 'roc_auc': 0.9437229437229436,
 'cv_roc_auc_best': 0.9028720876546965}

Inspect runs with `mlflow ui` (from the project root) at http://localhost:5000.